# 19)
We first consider proportional sampling. We define 4 intervals on $[0,1]$ as strata for $U \sim Unif(0,1)$

In [20]:
import numpy as np



def strata(U):
    if U < 0.25:
        return 0
    elif U < 0.4:
        return 1
    elif U < 0.8:
        return 2
    else:
        return 3




Since U is uniform, we have that the probabilities of each strata $p_i = P(Z = I)$ are given by
$$
p_0 = 0.25, p_1 = 0.15, p_2 = 0.4, p_3 = 0.2
$$

In [21]:
def strata_intervals(i):
    if i == 0:
        return (0, 0.25)
    elif i == 1:
        return (0.25, 0.4)
    elif i == 2:
        return (0.4, 0.8)
    else:
        return (0.8, 1)

M = 10000
S = 4

def proportional(M):
    p = np.array([
        np.diff(strata_intervals(i))[0]
        for i in range(S)
    ])

    N = np.round(M * p).astype(int)

    Ybar = np.zeros(S)
    sigma2 = np.zeros(S)
    for i in range(S):
        a, b = strata_intervals(i)
        # Generate N[i] observations specifically in stratum i
        U = np.random.uniform(a, b, N[i])
        Y = np.sqrt(1-U**2)
        Ybar[i] = np.mean(Y)
        sigma2[i] = np.sum((Y - Ybar[i]) ** 2) / (N[i] - 1)

    ls = np.sum(p * Ybar)
    sigma_str2 = np.sum(sigma2 * p ** 2 / N)
    return ls, sigma_str2, sigma2
ls, sigma_str2, _ = proportional(M)

For optimal sampling, we first run a small pilot run to estimate the variances of each strata using proportional allocation. We then use these variances to determine the optimal number of samples to take from each strata and estimate by optimal allocation.

In [22]:
_,_, sigma2 = proportional(1000)

def optimal_allocation(p,sigma2, M):
    rho = p*np.sqrt(sigma2) / np.sum(p * np.sqrt(sigma2))
    N = np.round(M * rho).astype(int)
    return rho, N

def optimal(M, sigma2):
    # Define probabilites for each strata
    p = np.array([
        np.diff(strata_intervals(i))[0]
        for i in range(S)
    ])
    # Compute optimal allocation
    rho, N = optimal_allocation(p, sigma2, M)

    Ybar = np.zeros(S)
    sigma2 = np.zeros(S)

    # Estimate
    for i in range(S):
        # Get interval for stratum i
        a, b = strata_intervals(i)
        # Generate N[i] observations specifically in stratum i
        U = np.random.uniform(a, b, N[i])
        Y = np.sqrt(1-U**2)

        Ybar[i] = np.mean(Y)
        sigma2[i] = np.sum((Y - Ybar[i]) ** 2) / (N[i] - 1)

    ls = np.sum(p * Ybar)
    sigma_str2 = np.sum(sigma2 * p ** 2 / N)
    return ls, sigma_str2, sigma2

ls_opt, sigma_str2_opt, _ = optimal(M, sigma2)

print("Variance reduction", sigma_str2 / sigma_str2_opt)

Variance reduction 1.5378026634923203
